# South Sudan Tabular Data

This notebook is used to prepare the location data to upload to Strapi.

The data model is as follows:
```typescript
interface Location {
  name: string; // Required
  type: 'administrative' | 'hydrological'; // Required
  level: number; // Required, must be 1, 2 or 3
  code: string; // Required, reference to the layer geometry
  parent: Location; // 1-to-1 relation to a parent location
}
```

The data can be exported as a JSON file with the following structure:
```json
{
  "version": 2,
  "data": {
    "api::location.location": {
      "1": {
        "id": 1,
        "name": "Level 1 example",
        "type": "administrative",
        "level": 1,
        "code": "01",
        "createdAt": "2024-10-28T13:40:23.054Z",
        "updatedAt": "2024-10-28T13:41:03.092Z",
        "parent": null,
        "createdBy": null,
        "updatedBy": null
      },
      "2": {
        "id": 2,
        "name": "Level 2 example",
        "type": "administrative",
        "level": 2,
        "code": "011",
        "createdAt": "2024-10-28T13:40:36.041Z",
        "updatedAt": "2024-10-28T13:40:59.185Z",
        "parent": 1,
        "createdBy": null,
        "updatedBy": null
      },
      "3": {
        "id": 3,
        "name": "Level 3 example",
        "type": "administrative",
        "level": 3,
        "code": "012",
        "createdAt": "2024-10-28T13:40:50.463Z",
        "updatedAt": "2024-10-28T13:40:50.463Z",
        "parent": 1,
        "createdBy": null,
        "updatedBy": null
      }
    }
  }
}
```




## Setup

### Library import

In [1]:
# imports
import json
import math
import re
import sys
from ast import literal_eval
from datetime import datetime
from pprint import pprint
from urllib.parse import quote

import pandas as pd
import requests
from google.cloud import storage

# Include local library paths if you have ../src/utils.py
sys.path.append("../src/")
sys.path.append("../src/animations")
sys.path.append("../src/datasets")
sys.path.append("../src/datasets/factory")
sys.path.append("../src/helpers")

from datasets.datasets import dataset_database

### Utils

In [2]:
def convert_string_to_float_list(s):
    """
    Convert a string of space-separated numbers to a list of floats.

    Parameters
    ----------
    s : str
        A string of space-separated numbers.
    """
    # Remove square brackets and newline characters
    s = s.strip("[]").replace("\n", "")

    # Split the string by spaces and filter out empty strings
    number_strings = re.split(r"\s+", s.strip())

    # Convert each number string to a float
    float_list = [float(num) for num in number_strings]

    return float_list

## Dataset information

In [3]:
datasets = dataset_database.datasets()
pprint(datasets)

{'Agricultural drought exposure': <datasets.datasets.Dataset object at 0x7f6f4aff5670>,
 'Agricultural drought hazard': <datasets.datasets.Dataset object at 0x7f6e89fdb3b0>,
 'Boundaries': <datasets.datasets.Dataset object at 0x7f6f63f0ffe0>,
 'Contextual layers': <datasets.datasets.Dataset object at 0x7f6e89b324e0>,
 'EO-based flood exposure': <datasets.datasets.Dataset object at 0x7f6e89b32870>,
 'EO-based flood hazard': <datasets.datasets.Dataset object at 0x7f6e89b329f0>,
 'Hydrographic data': <datasets.datasets.Dataset object at 0x7f6e89b328a0>,
 'Hydrometeorological Data': <datasets.datasets.Dataset object at 0x7f6e89b328d0>,
 'In-situ Data': <datasets.datasets.Dataset object at 0x7f6e89b32a80>,
 'Meteorological drought exposure': <datasets.datasets.Dataset object at 0x7f6e89b32ab0>,
 'Meteorological drought hazard': <datasets.datasets.Dataset object at 0x7f6e89b32ae0>,
 'Model-based flood exposure': <datasets.datasets.Dataset object at 0x7f6e89b32b10>,
 'Model-based flood hazard

## Load Layers

In [4]:
dataset = datasets["Boundaries"]
layers = dataset.layers()
layers

{'Administrative Boundaries - adm0': <datasets.datasets.Layer at 0x7f6e89b333e0>,
 'Administrative Boundaries - adm1': <datasets.datasets.Layer at 0x7f6e89b33380>,
 'Administrative Boundaries - adm2': <datasets.datasets.Layer at 0x7f6e89b334a0>,
 'Administrative Boundaries - adm3': <datasets.datasets.Layer at 0x7f6e89b33500>,
 'Hydrological Basins': <datasets.datasets.Layer at 0x7f6e89b33560>}

***
# Location data
## Process Data

In [7]:
layer_types = {
    "Administrative Boundaries - adm1": "administrative",
    "Administrative Boundaries - adm2": "administrative",
    "Administrative Boundaries - adm3": "administrative",
    "Hydrological Basins": "hydrological",
}

code_column = {
    "Administrative Boundaries - adm1": "adm1_pcode",
    "Administrative Boundaries - adm2": "adm2_pcode",
    "Administrative Boundaries - adm3": "adm3_pcode",
    "Hydrological Basins": "objectid",
}

name_column = {
    "Administrative Boundaries - adm1": "adm1_en",
    "Administrative Boundaries - adm2": "adm2_en",
    "Administrative Boundaries - adm3": "adm3_en",
    "Hydrological Basins": "basinname",
}

In [8]:
# Initialize the JSON structure
json_data = {"version": 2, "data": {"api::location.location": {}}}

# Dictionary to keep track of parent IDs
parent_ids = {}

location_id = 1
for layer_name, layer_type in layer_types.items():
    print(f"Processing {layer_name}")
    layer = layers[layer_name]
    df = layer.get_data()

    for _index, row in df.iterrows():
        code = str(row[code_column[layer_name]])
        if "Administrative Boundaries" in layer_name:
            level = int(layer_name.split("adm")[-1]) if "adm" in layer_name else 1
            parent_ids[code] = location_id
            if level > 1:
                try:
                    parent_id = parent_ids[code[:-2]]
                except KeyError:
                    parent_id = None
            else:
                parent_id = None
        else:
            level = 1
            parent_id = None

        location_data = {
            "id": location_id,
            "name": row[name_column[layer_name]],
            "type": layer_type,
            "level": level,
            "code": code,
            "createdAt": datetime.now().isoformat(),
            "updatedAt": datetime.now().isoformat(),
            "parent": parent_id,
            "createdBy": None,
            "updatedBy": None,
        }
        json_data["data"]["api::location.location"][str(location_id)] = location_data
        location_id += 1

# Convert to JSON string
json_string = json.dumps(json_data, indent=2)

# Write to file
with open("../data/processed/locations.json", "w") as f:
    f.write(json_string)

Processing Administrative Boundaries - adm1
Loading data from https://storage.googleapis.com/wbhydross_deliverables/D3-Database/00-%20Ancillary%20Layers/OCHA-SubnationalAdministrativeBoundaries/WBHYDROSSD_OCHA_SubnationalAdministrativeBoundaries-adm1_4326_SouthSudan_20230829_20240228.shp...
Processing Administrative Boundaries - adm2
Loading data from https://storage.googleapis.com/wbhydross_deliverables/D3-Database/00-%20Ancillary%20Layers/OCHA-SubnationalAdministrativeBoundaries/WBHYDROSSD_OCHA_SubnationalAdministrativeBoundaries-adm2_4326_SouthSudan_20230829_20240228.shp...
Processing Administrative Boundaries - adm3
Loading data from https://storage.googleapis.com/wbhydross_deliverables/D3-Database/00-%20Ancillary%20Layers/OCHA-SubnationalAdministrativeBoundaries/WBHYDROSSD_OCHA_SubnationalAdministrativeBoundaries-adm3_4326_SouthSudan_20230829_20240228.shp...
Processing Hydrological Basins
Loading data from https://storage.googleapis.com/wbhydross_deliverables/D3-Database/00-%20Anc

***
# Zonal statistics data
## Process Data

In [8]:
layer_ids = {"Evapotranspiration": 52, "Precipitation": 50, "Soil moisture": 53, "Temperature": 51}

code_columns = {
    "Administrative Boundaries - adm0": "adm0_pcode",
    "Administrative Boundaries - adm1": "adm1_pcode",
    "Administrative Boundaries - adm2": "adm2_pcode",
    "Administrative Boundaries - adm3": "adm3_pcode",
    "Hydrological Basins": "objectid",
}

In [9]:
keys_to_remove = ["Evapotranspiration"]
[layer_ids.pop(key, None) for key in keys_to_remove]

[52]

In [10]:
# Initialize the JSON structure
json_data = {"version": 2, "data": {"api::chart-data.chart-data": {}}}

chart_id = 1
for layer_name, layer_id in layer_ids.items():
    print(f"Processing {layer_name}")
    raster_name = layer_name.replace(" ", "_")
    for vector_name, code_column in code_columns.items():
        print(f"Processing {vector_name}")
        vector_name = (
            vector_name.lower()
            .replace(" - ", "_")
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
        )

        df = pd.read_csv(f"../data/processed/ZonalStatistics/{raster_name}_{vector_name}.csv")

        for _index, row in df.iterrows():
            code = str(row[code_column])

            chart_data = {
                "id": chart_id,
                "location_code": str(row[code_column]),
                "year": int(row["year"]),
                "x_values": literal_eval(row["x_axis_values"]),
                "y_values": convert_string_to_float_list(row["y_axis_values"]),
                "createdAt": datetime.now().isoformat(),
                "updatedAt": datetime.now().isoformat(),
                "layer": layer_id,
                "createdBy": None,
                "updatedBy": None,
            }
            json_data["data"]["api::chart-data.chart-data"][str(chart_id)] = chart_data
            chart_id += 1

# Convert to JSON string
json_string = json.dumps(json_data, indent=2)

# Write to file
with open("../data/processed/chart-data.json", "w") as f:
    f.write(json_string)

Processing Precipitation
Processing Administrative Boundaries - adm0
Processing Administrative Boundaries - adm1
Processing Administrative Boundaries - adm2
Processing Administrative Boundaries - adm3
Processing Hydrological Basins
Processing Soil moisture
Processing Administrative Boundaries - adm0
Processing Administrative Boundaries - adm1
Processing Administrative Boundaries - adm2
Processing Administrative Boundaries - adm3
Processing Hydrological Basins
Processing Temperature
Processing Administrative Boundaries - adm0
Processing Administrative Boundaries - adm1
Processing Administrative Boundaries - adm2
Processing Administrative Boundaries - adm3
Processing Hydrological Basins


***
# In-situ data
## Process Data

In [22]:
client = storage.Client(project="wb-hydro-ss")

bucket_name = "wbhydross_deliverables"
folder_paths = [
    "D3-Database/012-Data rescue/MWRI/MWRI_river_discharge_json/",
    "D3-Database/012-Data rescue/HYDROC/HYDROC_river_discharge_json/Vol1",
    "D3-Database/012-Data rescue/HYDROC/HYDROC_river_discharge_json/Vol2",
    "D3-Database/012-Data rescue/HYDROC/HYDROC_river_discharge_json/Vol3",
]

# Initialize the JSON structure
json_data = {"version": 2, "data": {"api::in-situ-data.in-situ-data": {}}}

point_id = 1
for folder_path in folder_paths:
    # List all the blobs in the specified folder
    bucket = client.get_bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=folder_path)

    # Filter the blobs to get only the .json files
    json_files = [blob for blob in blobs if blob.name.endswith(".json")]

    for json_file in json_files:
        public_url = quote(
            f"https://storage.googleapis.com/{bucket_name}/{json_file.name}", safe=":/"
        )

        # Fetch the content from the public URL
        response = requests.get(public_url)
        data = response.json()

        data_info = {}
        data_info["station"] = data["info"]["Station"]
        data_info["source"] = data["info"]["Source"]
        data_info["data_quality"] = data["info"]["Data Quality"]

        point_data = data_info

        x_values = [entry.get("index", entry.get("Time")) for entry in data["data"]]
        y_values = [entry["Values"] for entry in data["data"]]
        # Replace nan with None
        y_values = [None if math.isnan(value) else value for value in y_values]

        point_data["x_values"] = x_values
        point_data["y_values"] = y_values

        point_data["createdAt"] = datetime.now().isoformat()
        point_data["updatedAt"] = datetime.now().isoformat()
        point_data["createdBy"] = None
        point_data["updatedBy"] = None

        json_data["data"]["api::in-situ-data.in-situ-data"][str(point_id)] = point_data
        point_id += 1


# Convert to JSON string
json_string = json.dumps(json_data, indent=2)

# Write to file
with open("../data/processed/in-situ-data.json", "w") as f:
    f.write(json_string)